# Metriche goal recognition

Import vari

In [1]:
import pickle
import math
import re
import json
import sys 
from pathlib import Path
import pandas as pd
pd.set_option("display.precision", 2)
pd.set_option('display.max_rows', 200)

import numpy as np
from numpy.linalg import norm
import matplotlib.pyplot as plt

sys.path.append('/Users/riccardo/projects/goal_recognition/plan-image-conversion/src')
from plan import Plan # type: ignore
from action import Action # type: ignore

from embedding_dataset import EmbeddingSequence
from attention_extraction_layers import *
from keras.models import load_model

from dataclasses import dataclass
import pandas as pd
from IPython.display import display, Markdown
import pypandoc

In [2]:
def get_plan_info(test_plan):
    # estraggo oggetti
    objects = []
    for o in test_plan.initial_state:
        objects.extend(o.strip().split(' ')[1:])

    objects=set(objects)

    # estraggo goal
    goal = test_plan.goals
    actions = [action.name.lower() for action in test_plan.actions]
    return objects, goal, actions


def get_category_df(category):
    p_category_problems = []

    for p in problem_categories[category]: # type:ignore
        objects, goals, actions = get_plan_info(p)
        problem_name = p.plan_name.split('_')[0]
        problem_number = int(problem_name.replace('p', ''))
        p_category_problems.append({
            'obj_len':len(objects),
            'goal_len':len(goals),
            'actions_len':len(actions),
            'problem_number':problem_number
        })

    p_category_df = pd.DataFrame(p_category_problems)
    p_category_df.sort_values('problem_number', inplace=True)
    p_category_df.set_index('problem_number', inplace=True)
    return p_category_df


def get_categories_df(categories):
    p_category_problems = []
    for category in categories:

        for p in problem_categories[category]: # type:ignore
            objects, goals, actions = get_plan_info(p)
            problem_name = p.plan_name.split('_')[0]
            problem_number = int(problem_name.replace('p', ''))
            p_category_problems.append({
                'obj_len':len(objects),
                'goal_len':len(goals),
                'actions_len':len(actions),
                'problem_number':problem_number,
                'category':category
            })
    p_category_df = pd.DataFrame(p_category_problems)
    p_category_df.sort_values('problem_number', inplace=True)
    p_category_df.set_index('problem_number', inplace=True)
    return p_category_df

Elementi per l'analisi

In [9]:
EMBEDDING_DIM = 256
MAX_DIM = 100
NUM_CLASSES = 506

dizionario_goal = json.load(open('/Users/riccardo/projects/goal_recognition/visual-grnet/files/blocksworld/jsons/goal_indexes.json', 'r'))
all_goals = json.load(open('/Users/riccardo/projects/goal_recognition/visual-grnet/files/blocksworld/jsons/problem_goals.json', 'r'))
pergen_plan_groups = json.load(open('/Users/riccardo/projects/goal_recognition/visual-grnet/files/blocksworld/jsons/pergen_plans_groups.json', 'r'))
pergen_possible_goals = json.load(open('/Users/riccardo/projects/goal_recognition/visual-grnet/files/blocksworld/jsons/pergen_possible_goals.json', 'r'))

In [10]:
# Funzioni
def array_min_max(arr): # https://codepointtech.com/efficient-min-max-scaling-with-numpy-in-python/ 
    min_val = np.min(arr)
    max_val = np.max(arr)

    if (max_val - min_val) == 0:
        scaled_arr = np.full_like(arr, 0.5)
    else:
        scaled_arr = (arr - min_val) / (max_val - min_val)
    
    return scaled_arr

def get_score(prediction: np.ndarray, possible_goal: list):
    score = np.zeros(shape=(len(possible_goal)))
    for score_index, goal_index in enumerate(possible_goal):
        score[score_index] = prediction[int(goal_index)]
    return score

def get_max(scores: np.ndarray) -> list:
    max_element = -1
    index_max = list()
    for i in range(len(scores)):
        if scores[i] > max_element:
            max_element = scores[i]
            index_max = [i]
        elif scores[i] == max_element:
            index_max.append(i)

    return index_max

def get_test_sequences(test_npz_file, problem_names, percentage, mean, std, batch_size:int=1):
    """Genera la sequenza di embeddings usata per ottenere le predictions"""

    # get the embeddings and seq names
    all_test_embeddings = np.load(test_npz_file)
    all_test_seq_names = all_test_embeddings.get('seq_names')
    
    # shuffle
    random_shuffler = np.random.default_rng(seed=19024)
    random_shuffler.shuffle(all_test_seq_names)

    # genera il necessario per creare una EmbeddingSequence con dei piani specifici
    test_sequences_names = []
    test_sequences = {}

    for problem in problem_names:
        problem_name = problem['name']
        mask = np.char.startswith(all_test_seq_names, problem_name)
        sequences = all_test_seq_names[mask]
        sequence = sequences[0]
        
        test_sequences_names.append(sequence)
        test_sequences[f'embeddings_{sequence}'] = all_test_embeddings[f'embeddings_{sequence}']
        test_sequences[f'goal_{sequence}'] = all_test_embeddings[f'goal_{sequence}']
        
    
    test_sequences['seq_names'] = np.stack(test_sequences_names)
    return EmbeddingSequence(
        npz=test_sequences,
        max_dim=MAX_DIM,
        batch_size=batch_size,
        embedding_dim=EMBEDDING_DIM,
        num_classes=NUM_CLASSES,
        perc=percentage,
        norm_mean=mean,
        norm_std=std,
        ignore_last_n_states=0
    )

def get_results_df(category, percentage, confidence) ->tuple[pd.DataFrame, pd.DataFrame, list, list]:
    results = [] # risultati da trasformare in df
    scores = [] # contiene gli score del goal predetto e del goal corretto
    all_predictions = [] # tutte le predictions
    predicted_goals = [] # elementi del tipo (predicted_goals, correct_goal)
    comparisons = []

    test_problem_names = pergen_plan_groups[category]['plans']
    test_sequences = get_test_sequences(
        test_npz_file=npz_pergen_path,
        problem_names=test_problem_names,
        percentage=percentage,
        mean=None,
        std=None,
        batch_size=64
    )

    # ottengo le predictions
    predictions = model.predict(test_sequences, verbose=0) # type: ignore
    
    # ottengo i goal possibili per la categoria
    possible_goals = pergen_possible_goals.get(category, None)
    possible_goals_indexes = {}
    for goal_number, fluents in possible_goals.items():
        possible_goals_indexes[int(goal_number)] = [dizionario_goal.get(f.upper()) for f in fluents]
    
    # print(possible_goals_indexes)

    assert len(predictions) == len(test_problem_names), f'Problema con la lunghezza delle sequenze: {len(test_sequences.seq_names)} contro {len(test_problem_names)}'

    for pred, problem, dataset_seq_name in zip(predictions, test_problem_names, test_sequences.seq_names):
        y_pred = pred#array_min_max(pred)
        y_pred[y_pred<confidence] = 0
        problem_name = problem['name']
        assert dataset_seq_name == problem_name, f'Problema con i nomi delle sequenze, expected: {dataset_seq_name}, ottenuto: {problem_name}'
        correct_goal_number = int(problem['goal_number'])
        all_predictions.append(y_pred)

        scores_size = max([int(p) for p in possible_goals.keys()]) + 1
        problem_scores = np.zeros(shape=(scores_size))
        for goal_number, goal_indexes in sorted(possible_goals_indexes.items()):
            score = get_score(y_pred, goal_indexes)
            problem_scores[int(goal_number)] = np.sum(score)
            for fluent_score, fluent in zip(score, possible_goals[str(goal_number)]):
                results.append({
                    'fluente':fluent,
                    'id_fluente': dizionario_goal[fluent.upper()],
                    'fluent_score':fluent_score,
                    'goal_number':int(goal_number),
                    'correct_goal_number':int(correct_goal_number),
                    'categoria_fluente': fluent.split(' ')[0].strip(),
                    'sequenza_considerata':problem_name
                })
        # max_score_goals = get_max(problem_scores)
        # goal_idx = 0 if len(max_score_goals) == 1 else np.random.randint(0, len(max_score_goals))
        # predicted_goal = max_score_goals[goal_idx]
        predicted_goal = get_max(problem_scores)

        # predicted_goal = np.argsort(problem_scores).astype(int)[-3:]
        predicted_goals.append((correct_goal_number, predicted_goal))
        scores.append((problem_scores[correct_goal_number], problem_scores[predicted_goal]))
        comparisons.append({
            "correct_goal":correct_goal_number,
            "predicted_goal":predicted_goal,
            "correct_score":problem_scores[correct_goal_number],
            "predicted_score":problem_scores[predicted_goal],
            "difference":problem_scores[predicted_goal]-problem_scores[correct_goal_number],
            "problem_name":problem_name
        })

    res = pd.DataFrame(results)
    comparisons_df = pd.DataFrame(comparisons)
    return res, comparisons_df, predicted_goals, scores

def get_fluent_analysis(res_df):
    res_dataframe = res_df.sort_values(['fluente', 'sequenza_considerata']).drop_duplicates(subset=['fluente', 'sequenza_considerata'], keep='first')
    fluent_groups = res_dataframe.groupby(['fluente', 'id_fluente'])['fluent_score']

    fluent_means = fluent_groups.mean()
    fluent_std = fluent_groups.std()
    fluent_min = fluent_groups.min()
    fluent_max = fluent_groups.max()

    fluent_df = pd.concat([fluent_means, fluent_std, fluent_min, fluent_max], axis=1)
    fluent_df.columns = ['media', 'std', 'minimo', 'massimo']
    return fluent_df.sort_values('media', ascending=False)

def get_correctness_df(predicted_goals):
    correct_dict = dict()
    predicted_dict = dict()
    predicted_correct_dict = dict()

    for corretto, predicted in predicted_goals:
        if corretto not in correct_dict.keys():
            correct_dict[corretto] = 1
        else:
            correct_dict[corretto] += 1

        if predicted not in predicted_dict.keys():
            predicted_dict[predicted] = 1
        else:
            predicted_dict[predicted] += 1

        if corretto == predicted:
            if predicted not in predicted_correct_dict.keys():
                predicted_correct_dict[predicted] = 1
            else:
                predicted_correct_dict[predicted] += 1

    correctness_analysis = {
        'corretti':correct_dict,
        'predicted':predicted_dict,
        'predicted_corretti':predicted_correct_dict
    }

    correctness_df = pd.DataFrame(correctness_analysis)
    correctness_df = correctness_df.sort_index().fillna(0)

    predetti_corretti = int(correctness_df['predicted_corretti'].sum())
    predetti = int(correctness_df['predicted'].sum())

    print(f'Accuracy: {predetti_corretti/predetti*100:.2f}% ({predetti_corretti}/{predetti})')
    return correctness_df

def plot_correctness_df(correctness_df:pd.DataFrame):
    ax = correctness_df.plot(kind='bar', figsize=(12, 6))

    for container in ax.containers:
        ax.bar_label(container, fontsize=8) #type: ignore

    plt.title(category)
    plt.xlabel('Goal Number')
    plt.xticks(rotation=0)
    plt.show()

def calculate_accuracy(results:list[tuple]):
    correct_score = 0
    for correct, predicted in results:
        if correct in predicted:
            correct_score += 1/len(predicted)
    
    print(f'Accuracy: {correct_score/len(results)*100:.2f}%')


In [11]:
def get_detailed_scores(category, percentage, confidence, model):
    results = []

    test_problem_names = pergen_plan_groups[category]['plans']
    test_sequences = get_test_sequences(
        test_npz_file=npz_pergen_path,
        problem_names=test_problem_names,
        percentage=percentage,
        mean=None,
        std=None,
        batch_size=128
    )

    # ottengo le predictions
    predictions = model.predict(test_sequences, verbose=0) # type: ignore
    predictions[predictions<confidence] = 0
    
    # ottengo i goal possibili per la categoria
    possible_goals = pergen_possible_goals.get(category, None)
    possible_goals_indexes = {}
    for goal_number, fluents in possible_goals.items():
        possible_goals_indexes[int(goal_number)] = [dizionario_goal.get(f.upper()) for f in fluents]

    assert len(predictions) == len(test_problem_names), f'Problema con la lunghezza delle sequenze: {len(test_sequences.seq_names)} contro {len(test_problem_names)}'
    scores_size = max([int(p) for p in possible_goals.keys()]) + 1

    for y_pred, problem, dataset_seq_name in zip(predictions, test_problem_names, test_sequences.seq_names):
        problem_name = problem['name']
        assert dataset_seq_name == problem_name, f'Problema con i nomi delle sequenze, expected: {dataset_seq_name}, ottenuto: {problem_name}'
        correct_goal_number = int(problem['goal_number'])

        problem_scores = np.zeros(shape=(scores_size))
        for goal_number, goal_indexes in sorted(possible_goals_indexes.items()):
            score = get_score(y_pred, goal_indexes)
            problem_scores[int(goal_number)] = np.sum(score)

        results.append({
            "problem":problem_name,
            "correct_goal":correct_goal_number,
            "scores":problem_scores
        })
    
    return results

Calcolo delle predictions in maniera dettagliata

`predicions_1.pkl`:

```python
versions = [
    [417, 419],
    [704, 705],
    [806, 805]
    ]
titles = ['SAE', 'DAE', 'VAE']
adapt = ['Con Adapter', 'Senza Adapter']
```

In [ ]:
categories = [f'p0{i}' for i in range(1,8)]
percentages = [0.1,0.3,0.5,0.7,1.0]
confidences = [0]
categories_baseline = {
    'p01' : 21,
    'p02' : 20,
    'p03' : 19,
    'p04' : 20,
    'p05' : 20,
    'p06' : 20,
    'p07' : 19,
}

cached_predictions_path = Path("/Users/riccardo/projects/goal_recognition/visual-grnet/files/blocksworld/predictions_cache/predictions_1.pkl")
if cached_predictions_path.exists():
    saved_predictions = pickle.load(open(cached_predictions_path, "rb"))
    versions = saved_predictions.get("versions")
    versions_scores = saved_predictions.get("version_scores")
    versions_scores_without_category = saved_predictions.get("versions_scores_without_category")
else:
    versions_scores = {}
    versions_scores_without_category = {}
    versions = [
        (420, "sae"),
        (421, "sae-ablato"),

        # da fare train
        (704, "dae"), 
        (705, "dae-ablato"),
        (806, "vae"),
        (805, "vae-ablato")
    ]

    for version, embeddings_type in versions:
        model_path = Path(f"/Users/riccardo/projects/goal_recognition/visual-grnet/files/blocksworld/experiments/{version}/visual-grnet-bw-v{version}.keras")
        npz_pergen_path = Path(f"/Users/riccardo/projects/goal_recognition/visual-grnet/files/blocksworld/embeddings_cache/{embeddings_type.split("-")[0]}/test_pergen.npz")
        model = load_model(
                model_path,
                custom_objects={
                    "AttentionWeights": AttentionWeights,
                    "ContextVector": ContextVector
                },
                compile=False
            )

        detailed_scores = {}
        for percentage in percentages:
            for category in categories:
                for confidence in confidences:
                    detailed_scores[(category, percentage, confidence)] = get_detailed_scores(category, percentage, confidence, model)

        detailed_scores_without_category = dict()
        for key, scores in detailed_scores.items():
            _, percentage, confidence = key
            if (percentage, confidence) not in detailed_scores_without_category.keys():
                detailed_scores_without_category[(percentage, confidence)] = [scores]
            else:
                detailed_scores_without_category[(percentage, confidence)].append(scores)

        versions_scores[embeddings_type] = detailed_scores
        versions_scores_without_category[embeddings_type] = detailed_scores_without_category


    obj_to_save = {
        "versions":versions,
        "version_scores":versions_scores,
        "versions_scores_without_category":versions_scores_without_category
    }

    pickle.dump(obj_to_save, open(cached_predictions_path, "wb"))

Calcolo delle accuracy con confidenza

In [13]:
def get_accuracies_df(detailed_scores, embeddings_type):
    accuracies = []

    for percentage in percentages:
        for category in categories:
            for confidence in confidences:
                correct = 0.0
                percentage_scores = detailed_scores[(category, percentage, confidence)]
                for detailed_score in percentage_scores:
                    correct_goal = detailed_score.get("correct_goal")
                    scores = detailed_score.get("scores")
                    predicted_goal = get_max(scores) # type: ignore

                    if correct_goal in predicted_goal:
                        correct += 1/len(predicted_goal)

                    accuracy = 100*correct/len(percentage_scores)
                    
                accuracies.append({
                    "category":category,
                    "percentage":percentage,
                    "accuracy":accuracy,
                    "embeddings_type":embeddings_type
                    # "confidence":confidence
                })

    accuracies_df = pd.DataFrame(accuracies).set_index("percentage")
    return accuracies_df


def get_cumulative_accuracies_df(detailed_scores_without_category, embeddings_type):
    cumulative_accuracies = []

    for percentage in percentages:
        for confidence in confidences:
            correct = 0.0
            percentage_scores = detailed_scores_without_category[(percentage, confidence)]
            total = 0
            for percentage_score in percentage_scores:
                for detailed_score in percentage_score:
                    correct_goal = detailed_score.get("correct_goal")
                    scores = detailed_score.get("scores")
                    predicted_goal = get_max(scores) # type: ignore

                    if correct_goal in predicted_goal:
                        correct += 1/len(predicted_goal)
                    
                    total+=1

            accuracy = 100*correct/total
                
            cumulative_accuracies.append({
                "percentage":percentage,
                "accuracy":accuracy,
                "embeddings_type":embeddings_type,
                # "confidence":confidence
            })

    cumulative_accuracies_df = pd.DataFrame(cumulative_accuracies).set_index("percentage").rename(
        index={p:int(p*100) for p in percentages},
        columns={c:f"{c:.2f}" for c in confidences}
    )
    return cumulative_accuracies_df

tabella latex delle gr-acc dettagliate

In [42]:
def format_and_bold_max(df):
    # Creiamo una copia del dataframe convertita a object per inserire le stringhe
    df_fmt = df.copy().astype(object)
    
    # Identifichiamo la riga della baseline (per escluderla dal confronto)
    mask_baseline = df.index.get_level_values(0) == 'baseline'
    
    # Formattiamo la baseline normalmente (senza grassetto)
    df_fmt.loc[mask_baseline] = df.loc[mask_baseline].map(lambda x: f"${x:.2f}$")
    
    # Estraiamo le percentuali (escludendo il '-' della baseline)
    percentages = [p for p in df.index.get_level_values(1).unique() if p != '-']
    # Estraiamo le categorie (livello interno delle colonne, es. p00, p01...)
    categories = df.columns.get_level_values(1).unique()
    
    # Cicliamo su ogni singola percentuale
    for pct in percentages:
        # Filtriamo le righe per la percentuale corrente (solo SAE, DAE, VAE)
        mask_pct = (df.index.get_level_values(1) == pct) & (~mask_baseline)
        
        # Cicliamo su ogni categoria
        for cat in categories:
            # Selezioniamo le colonne della categoria corrente (Con e Senza Adapter)
            mask_cat = df.columns.get_level_values(1) == cat
            
            # Estraiamo il blocco 3x2 di valori numerici
            subset = df.loc[mask_pct, mask_cat]
            
            # Troviamo il valore massimo assoluto nel blocco
            max_val = subset.max().max()
            
            # Applichiamo la formattazione a ciascuna cella del blocco
            for idx in subset.index:
                for col in subset.columns:
                    val = df.loc[idx, col]
                    
                    # Confronto con tolleranza per evitare problemi coi floating point
                    if abs(val - max_val) < 1e-6:
                        # Grassetto matematico per il valore massimo
                        df_fmt.loc[idx, col] = f"$\\mathbf{{{val:.2f}}}$"
                    else:
                        # Formattazione normale per gli altri valori
                        df_fmt.loc[idx, col] = f"${val:.2f}$"
                        
    return df_fmt

def format_and_bold_max_cumulative(df):
    # Copia e conversione a object per le stringhe LaTeX
    df_fmt = df.copy().astype(object)
    
    # Maschera e formattazione per la baseline
    mask_baseline = df.index.get_level_values(0) == 'baseline'
    df_fmt.loc[mask_baseline] = df.loc[mask_baseline].map(lambda x: f"${x:.2f}$")
    
    # Estrazione delle percentuali
    percentages = [p for p in df.index.get_level_values(1).unique() if p != '-']
    
    for pct in percentages:
        # Isoliamo le righe relative alla percentuale corrente (SAE, DAE, VAE)
        mask_pct = (df.index.get_level_values(1) == pct) & (~mask_baseline)
        
        # Estraiamo il blocco (che in questo caso sarà una griglia 3 righe x 2 colonne)
        subset = df.loc[mask_pct]
        
        # Troviamo il valore massimo assoluto nel blocco 3x2
        max_val = subset.max().max()
        
        # Applichiamo la formattazione
        for idx in subset.index:
            for col in subset.columns:
                val = df.loc[idx, col]
                
                # Applichiamo \mathbf al valore massimo, tolleranza sui float
                if abs(val - max_val) < 1e-6:
                    df_fmt.loc[idx, col] = f"$\\mathbf{{{val:.2f}}}$"
                else:
                    df_fmt.loc[idx, col] = f"${val:.2f}$"
                    
    return df_fmt


In [47]:
accuracies_df = pd.concat({
    "Con Adapter" : pd.concat([get_accuracies_df(versions_scores[v], v) for _,v in versions if not("ablato" in v)], axis=0).reset_index().pivot(
        index=["embeddings_type", "percentage"], columns="category"
    ),
    "Senza Adapter" : pd.concat([get_accuracies_df(versions_scores[v], v.split("-")[0]) for _,v in versions if ("ablato" in v)], axis=0).reset_index().pivot(
        index=["embeddings_type", "percentage"], columns="category"
    ),
}, axis=1)

accuracies_df.columns = pd.MultiIndex.from_tuples([(c[0],c[2]) for c in accuracies_df.columns])

custom_order = ["sae", "dae", "vae"]
level_0 = pd.CategoricalIndex(
    accuracies_df.index.get_level_values("embeddings_type"),
    categories=custom_order,
    ordered=True,
)
level_1 = accuracies_df.index.get_level_values("percentage")
accuracies_df.index = pd.MultiIndex.from_arrays([level_0, level_1], names=accuracies_df.index.names)
accuracies_df = accuracies_df.sort_index()
chiave_nuova_riga = ('baseline', "-")
nuova_riga = pd.DataFrame(
    data=[[100.0/p for p in categories_baseline.values()] * 2],
    index=pd.MultiIndex.from_tuples([chiave_nuova_riga], names=accuracies_df.index.names),
    columns=accuracies_df.columns
)
accuracies_df = pd.concat([nuova_riga, accuracies_df], axis=0)

# latexify
# accuracies_df = accuracies_df.map(lambda x: f"${x:.2f}$")
# accuracies_df.index.names = ["\\textbf{Embeddings}", "\\textbf{Oss. (\\%)}"]
# latexify
accuracies_df = format_and_bold_max(accuracies_df)
accuracies_df.index.names = ["\\textbf{Embeddings}", "\\textbf{Oss. (\\%)}"]
accuracies_df = accuracies_df.rename(
    index={p:int(p*100) for p in percentages},
    columns={v[1]:f"\\textbf{{{v[1].upper()}}}" for v in versions}
).rename(
    columns={f"p0{i}":f"\\textit{{C.{i}}}" for i in range(8)},
    index={v:f"\\textbf{{{v.upper()}}}" for v in list(set([a[0] for a in accuracies_df.index.values]))}
).rename(
    columns={
        "Con Adapter" : "\\textbf{{Con Adapter}}",
        "Senza Adapter" : "\\textbf{{Senza Adapter}}",
    },
    index={int(p*100):f"${int(p*100)}$" for p in percentages}
)

latex_acc_df = accuracies_df.to_latex(float_format="%.2f", column_format="cc|ccccccc|ccccccc", multirow=True)
latex_acc_df = latex_acc_df.replace(r"\multicolumn{7}{r}{\textbf{{Con Adapter}}}", r"\multicolumn{7}{c|}{\textbf{{Con Adapter}}}")
latex_acc_df = latex_acc_df.replace(r"\multicolumn{7}{r}", r"\multicolumn{7}{c}")
latex_acc_df = latex_acc_df.replace(r"\multicolumn{7}{r}", r"\multicolumn{7}{c}")

latex_acc_df = latex_acc_df.replace('\\cline{1-16}\n', '')
# latex_acc_df = latex_acc_df.replace('\\midrule\n', '')  
latex_acc_df = latex_acc_df.replace(r'\multirow[t]', '\\midrule\n\\multirow')

pattern = r"^\s*&\s*&\s*(.*?)\s*\\\\\n\s*(.*?)\s*(?:&\s*)+\\\\"
latex_acc_df = re.sub(pattern, r"\2 & \1 \\\\", latex_acc_df, flags=re.MULTILINE)

print(latex_acc_df)

\begin{tabular}{cc|ccccccc|ccccccc}
\toprule
 &  & \multicolumn{7}{c|}{\textbf{{Con Adapter}}} & \multicolumn{7}{c}{\textbf{{Senza Adapter}}} \\
\textbf{Embeddings} & \textbf{Oss. (\%)} & \textit{C.1} & \textit{C.2} & \textit{C.3} & \textit{C.4} & \textit{C.5} & \textit{C.6} & \textit{C.7} & \textit{C.1} & \textit{C.2} & \textit{C.3} & \textit{C.4} & \textit{C.5} & \textit{C.6} & \textit{C.7} \\
\midrule
\textbf{BASELINE} & - & $4.76$ & $5.00$ & $5.26$ & $5.00$ & $5.00$ & $5.00$ & $5.26$ & $4.76$ & $5.00$ & $5.26$ & $5.00$ & $5.00$ & $5.00$ & $5.26$ \\
\midrule
\multirow{5}{*}{\textbf{SAE}} & $10$ & $10.00$ & $10.20$ & $\mathbf{14.60}$ & $6.40$ & $6.45$ & $7.40$ & $11.00$ & $11.40$ & $11.40$ & $10.00$ & $5.20$ & $3.63$ & $8.00$ & $5.00$ \\
 & $30$ & $21.00$ & $23.00$ & $25.40$ & $\mathbf{21.80}$ & $\mathbf{26.81}$ & $23.20$ & $45.40$ & $12.80$ & $18.40$ & $16.60$ & $9.40$ & $11.49$ & $10.40$ & $4.80$ \\
 & $50$ & $41.60$ & $36.20$ & $\mathbf{43.00}$ & $\mathbf{45.00}$ & $48.39$ & $52.6

tabella latex delle gr-acc aggregate

In [48]:
cumulative_accuracies_df = pd.concat({
    "Con Adapter":pd.concat([get_cumulative_accuracies_df(versions_scores_without_category[v], v) for _,v in versions if not("ablato" in v)], axis=0).reset_index().set_index(["embeddings_type", "percentage"]),
    "Senza Adapter":pd.concat([get_cumulative_accuracies_df(versions_scores_without_category[v], v.split("-")[0]) for _,v in versions if ("ablato" in v)], axis=0).reset_index().set_index(["embeddings_type", "percentage"])
}, axis=1)
cumulative_accuracies_df.columns = [c[0] for c in cumulative_accuracies_df.columns]

custom_order = ["sae", "dae", "vae"]
level_0 = pd.CategoricalIndex(
    cumulative_accuracies_df.index.get_level_values("embeddings_type"),
    categories=custom_order,
    ordered=True,
)
level_1 = cumulative_accuracies_df.index.get_level_values("percentage")
cumulative_accuracies_df.index = pd.MultiIndex.from_arrays([level_0, level_1], names=cumulative_accuracies_df.index.names)
cumulative_accuracies_df = cumulative_accuracies_df.sort_index()

chiave_nuova_riga = ('baseline', "-")
media_baseline = sum(categories_baseline.values()) / len(categories_baseline.values())
nuova_riga = pd.DataFrame(
    data=[[100.0/media_baseline] * 2],
    index=pd.MultiIndex.from_tuples([chiave_nuova_riga], names=cumulative_accuracies_df.index.names),
    columns=cumulative_accuracies_df.columns
)
cumulative_accuracies_df = pd.concat([nuova_riga, cumulative_accuracies_df], axis=0)

# cumulative_accuracies_df = cumulative_accuracies_df.map(lambda x: f"${x:.2f}$")
cumulative_accuracies_df = format_and_bold_max_cumulative(cumulative_accuracies_df)
cumulative_accuracies_df.index.names = ["\\textbf{Embeddings}", "\\textbf{Oss. (\\%)}"]
cumulative_accuracies_df = cumulative_accuracies_df.rename(
    index={p:int(p*100) for p in percentages},
    columns={v[1]:f"\\textbf{{{v[1].upper()}}}" for v in versions}
).rename(
    columns={f"p0{i}":f"\\textit{{C.{i}}}" for i in range(8)},
    index={v:f"\\textbf{{{v.upper()}}}" for v in list(set([a[0] for a in cumulative_accuracies_df.index.values]))}
).rename(
    columns={
        "Con Adapter" : "\\textbf{{Con Adapter}}",
        "Senza Adapter" : "\\textbf{{Senza Adapter}}",
    },
    index={int(p*100):f"${int(p*100)}$" for p in percentages}
)

cumulative_latex_acc_df = cumulative_accuracies_df.to_latex(float_format="%.2f", column_format="cc|c|c", multirow=True)

cumulative_latex_acc_df = cumulative_latex_acc_df.replace('\\cline{1-4}\n', '')
# cumulative_latex_acc_df = cumulative_latex_acc_df.replace('\\midrule\n', '')  
cumulative_latex_acc_df = cumulative_latex_acc_df.replace(r'\multirow[t]', '\\midrule\n\\multirow')

pattern = r"^\s*&\s*&\s*(.*?)\s*\\\\\n\s*(.*?)\s*(?:&\s*)+\\\\"
cumulative_latex_acc_df = re.sub(pattern, r"\2 & \1 \\\\", cumulative_latex_acc_df, flags=re.MULTILINE)

print(cumulative_latex_acc_df)
# cumulative_accuracies_df

\begin{tabular}{cc|c|c}
\toprule
\textbf{Embeddings} & \textbf{Oss. (\%)} & \textbf{{Con Adapter}} & \textbf{{Senza Adapter}} \\
\midrule
\textbf{BASELINE} & - & $5.04$ & $5.04$ \\
\midrule
\multirow{5}{*}{\textbf{SAE}} & $10$ & $9.44$ & $7.81$ \\
 & $30$ & $26.66$ & $11.99$ \\
 & $50$ & $48.86$ & $17.11$ \\
 & $70$ & $\mathbf{70.45}$ & $25.23$ \\
 & $100$ & $77.26$ & $31.95$ \\
\midrule
\multirow{5}{*}{\textbf{DAE}} & $10$ & $9.84$ & $7.81$ \\
 & $30$ & $\mathbf{28.09}$ & $12.61$ \\
 & $50$ & $\mathbf{50.11}$ & $19.14$ \\
 & $70$ & $\mathbf{70.45}$ & $27.09$ \\
 & $100$ & $\mathbf{78.03}$ & $33.95$ \\
\midrule
\multirow{5}{*}{\textbf{VAE}} & $10$ & $\mathbf{9.93}$ & $7.87$ \\
 & $30$ & $26.54$ & $12.81$ \\
 & $50$ & $49.51$ & $21.28$ \\
 & $70$ & $68.82$ & $30.81$ \\
 & $100$ & $77.35$ & $40.16$ \\
\bottomrule
\end{tabular}



Calcolo theta accuracy e spread

In [52]:
theta_confidence = 0
thetas = [0, 0.1, 0.2]

def get_theta_accuracy_df(detailed_scores):    
    values = []
    for percentage in percentages:
        for category in categories:
            for theta in thetas:
                percentage_scores = detailed_scores[(category, percentage, theta_confidence)]
                threshold = 1-theta
                correct = 0
                spread = 0

                for detailed_score in percentage_scores:
                    correct_goal = detailed_score.get("correct_goal")
                    scores = detailed_score.get("scores")
                    scaled_scores = array_min_max(scores)
                    valid_scores = np.where(scaled_scores>=threshold, scaled_scores, 0)
                    goal_set = np.where(valid_scores>0)[0]

                    spread+=len(goal_set)
                    if correct_goal in goal_set:
                        correct+=1

                theta_acc = 100*correct/len(percentage_scores)
                spread_metric = spread/len(percentage_scores)

                values.append({
                    "category":category,
                    "percentage":percentage,
                    "theta":theta,
                    "theta_acc":theta_acc,
                    "spread":spread_metric
                })

    theta_accuracy_df = pd.DataFrame(values).pivot(index=["percentage", "category"], columns="theta", values=["theta_acc", "spread"]).rename(
        index={
            p:int(p*100) for p in percentages
        },
        columns={
            "theta_acc":r"$\theta$-accuracy",
            "spread":"Spread",
            0.0:"0",
            0.1:"0.1",
            0.2:"0.2",
        }
    )

    return theta_accuracy_df

def get_cumulative_theta_accuracies_df(detailed_scores_without_category):
    cumulative_values = []
    for percentage in percentages:
        for theta in thetas:
            percentage_scores = detailed_scores_without_category[(percentage, theta_confidence)]
            threshold = 1-theta
            correct = 0
            spread = 0
            total = 0

            for percentage_score in percentage_scores:
                for detailed_score in percentage_score:
                    correct_goal = detailed_score.get("correct_goal")
                    scores = detailed_score.get("scores")
                    scaled_scores = array_min_max(scores)
                    valid_scores = np.where(scaled_scores>=threshold, scaled_scores, 0)
                    goal_set = np.where(valid_scores>0)[0]

                    spread+=len(goal_set)
                    if correct_goal in goal_set:
                        correct+=1
                    total+=1

            theta_acc = 100*correct/total
            spread_metric = spread/total

            cumulative_values.append({
                "percentage":percentage,
                "theta":theta,
                "theta_acc":theta_acc,
                "spread":spread_metric
            })

    cumulative_theta_accuracy_df = pd.DataFrame(cumulative_values).pivot(index="percentage", columns="theta", values=["theta_acc", "spread"]).rename(
        index={
            p:int(p*100) for p in percentages
        },
        columns={
            "theta_acc":r"$\theta$-accuracy",
            "spread":"Spread"
        }
    )

    return cumulative_theta_accuracy_df

tabella latex delle $\theta$-acc dettagliate

In [ ]:
def format_theta_metrics(df):
    # Creiamo una copia convertita a object per le stringhe LaTeX
    df_fmt = df.copy().astype(object)
    
    # Estraiamo dinamicamente i livelli delle colonne
    models = df.columns.get_level_values(0).unique()    # es. sae, dae, vae
    metrics = df.columns.get_level_values(1).unique()   # es. \theta-accuracy, Spread
    thetas = df.columns.get_level_values(2).unique()    # es. 0, 0.1, 0.2
    
    # Iteriamo su ogni riga del DataFrame
    for idx in df.index:
        # Iteriamo su ogni metrica e su ogni valore di theta
        for metric in metrics:
            for theta in thetas:
                
                # Costruiamo la lista delle 3 colonne da confrontare per questa riga
                cols_to_compare = [(m, metric, theta) for m in models]
                
                # Estraiamo i valori numerici associati a queste 3 celle
                vals = df.loc[idx, cols_to_compare].astype(float)
                
                # Scegliamo min o max in base al nome della metrica
                metric_str = str(metric).lower()
                if 'spread' in metric_str:
                    target_val = vals.min()
                else:
                    target_val = vals.max()
                    
                # Applichiamo la formattazione in grassetto al valore target
                for col in cols_to_compare:
                    val = df.loc[idx, col]
                    
                    # Usiamo una tolleranza di 1e-6 per un confronto sicuro tra float
                    if pd.notna(val) and abs(val - target_val) < 1e-6:
                        df_fmt.loc[idx, col] = f"$\\mathbf{{{val:.2f}}}$"
                    else:
                        df_fmt.loc[idx, col] = f"${val:.2f}$"
                        
    return df_fmt


import pandas as pd

def format_cumulative_theta_metrics(df):
    # Creiamo una copia convertita a object per inserire i tag LaTeX
    df_fmt = df.copy().astype(object)
    
    # Controllo di sicurezza per un'eventuale riga 'baseline'
    mask_baseline = df.index.get_level_values(0) == 'baseline'
    if mask_baseline.any():
        df_fmt.loc[mask_baseline] = df.loc[mask_baseline].map(lambda x: f"${x:.2f}$")
    
    # Estraiamo le percentuali uniche dal livello 1 dell'indice (es. 10, 30, 50...)
    percentages = [p for p in df.index.get_level_values(1).unique() if p != '-']
    
    # Cicliamo su ogni singola percentuale
    for pct in percentages:
        # Maschera per isolare i modelli a quella data percentuale
        mask_pct = (df.index.get_level_values(1) == pct) & (~mask_baseline)
        
        # Cicliamo su ogni colonna (Metrica, Theta)
        for col in df.columns:
            metric = col[0] # Il primo livello della colonna contiene la metrica
            
            # Estraiamo i valori dei modelli in quella colonna e percentuale
            vals = df.loc[mask_pct, col].astype(float)
            
            # Scegliamo min o max in base al nome della metrica
            metric_str = str(metric).lower()
            if 'spread' in metric_str:
                target_val = vals.min()
            else:
                target_val = vals.max()
                
            # Applichiamo la formattazione
            for idx in df.loc[mask_pct].index:
                val = df.loc[idx, col]

                # Confronto float con tolleranza
                if pd.notna(val) and abs(val - target_val) < 1e-6:
                    df_fmt.loc[idx, col] = f"$\\mathbf{{{val:.2f}}}$"
                else:
                    df_fmt.loc[idx, col] = f"${val:.2f}$"
                    
    return df_fmt

In [60]:
theta_accuracy_df = pd.concat({v:get_theta_accuracy_df(versions_scores[v]) for _,v in versions if not("ablato" in v)}, axis=1)

theta_accuracy_df.index.names = [r"\textbf{\%Oss.}", r"\textbf{Cat}"]
theta_accuracy_df.columns.names = [None, None, None]
theta_accuracy_df = theta_accuracy_df.rename(
    index={f"p0{i}":f"\\textit{{C.{i}}}" for i in range(8)},
    columns={c[1]:f"\\textbf{{{c[1]}}}" for c in theta_accuracy_df.columns}
    ).rename(
        index={int(p*100):f"${int(p*100)}$" for p in percentages},
        columns={
            r"$\theta$-accuracy": r"\textbf{$\theta$-accuracy}",
            "Spread": r"\textbf{Spread}"
        }
    ).rename(
        columns={v[1]:f"\\textbf{{{v[1].upper()}}}" for v in versions}
    ).rename(
        columns={f"{t}":f"${t}$" for t in thetas}
    )

# theta_accuracy_df = theta_accuracy_df.map(lambda x: f"${x:.2f}$")
theta_accuracy_df = format_theta_metrics(theta_accuracy_df)

latex_theta_acc_df = theta_accuracy_df.to_latex(float_format="%.2f", column_format="cc|ccc|ccc|ccc|ccc|ccc|ccc", multirow=True, escape=False)
latex_theta_acc_df = latex_theta_acc_df.replace('\\cline{1-20}\n', '')
latex_theta_acc_df = latex_theta_acc_df.replace('\\midrule\n', '')
latex_theta_acc_df = latex_theta_acc_df.replace(r'\multirow[t]', '\\midrule\n\\multirow')
latex_theta_acc_df = latex_theta_acc_df.replace(r'\multicolumn{6}{r}', r'\multicolumn{6}{c|}')
latex_theta_acc_df = latex_theta_acc_df.replace(r'\multicolumn{3}{r}{\textbf{Spread}}', r'\multicolumn{3}{c|}{\textbf{Spread}}')
latex_theta_acc_df = latex_theta_acc_df.replace(r'\multicolumn{3}{r}', r'\multicolumn{3}{c}')

latex_theta_acc_df = latex_theta_acc_df.replace(r'\multicolumn{3}{c|}{\textbf{Spread}} \\', r'\multicolumn{3}{c}{\textbf{Spread}} \\')
latex_theta_acc_df = latex_theta_acc_df.replace(r'\multicolumn{6}{c|}{\textbf{VAE}}', r'\multicolumn{6}{c}{\textbf{VAE}}')

pattern = r"^\s*&\s*&\s*(.*?)\s*\\\\\n\s*(.*?)\s*(?:&\s*)+\\\\"
latex_theta_acc_df = re.sub(pattern, r"\2 & \1 \\\\", latex_theta_acc_df, flags=re.MULTILINE)

print(latex_theta_acc_df)

\begin{tabular}{cc|ccc|ccc|ccc|ccc|ccc|ccc}
\toprule
 &  & \multicolumn{6}{c|}{\textbf{SAE}} & \multicolumn{6}{c|}{\textbf{DAE}} & \multicolumn{6}{c}{\textbf{VAE}} \\
 &  & \multicolumn{3}{c}{\textbf{$\theta$-accuracy}} & \multicolumn{3}{c|}{\textbf{Spread}} & \multicolumn{3}{c}{\textbf{$\theta$-accuracy}} & \multicolumn{3}{c|}{\textbf{Spread}} & \multicolumn{3}{c}{\textbf{$\theta$-accuracy}} & \multicolumn{3}{c}{\textbf{Spread}} \\
\textbf{\%Oss.} & \textbf{Cat} & $0$ & $0.1$ & $0.2$ & $0$ & $0.1$ & $0.2$ & $0$ & $0.1$ & $0.2$ & $0$ & $0.1$ & $0.2$ & $0$ & $0.1$ & $0.2$ & $0$ & $0.1$ & $0.2$ \\
\midrule
\multirow{7}{*}{$10$} & \textit{C.1} & $10.00$ & $20.60$ & $28.80$ & $\mathbf{1.00}$ & $\mathbf{2.05}$ & $\mathbf{2.87}$ & $11.20$ & $\mathbf{23.40}$ & $\mathbf{30.20}$ & $\mathbf{1.00}$ & $2.08$ & $2.90$ & $\mathbf{13.80}$ & $\mathbf{23.40}$ & $28.20$ & $\mathbf{1.00}$ & $2.17$ & $2.90$ \\
 & \textit{C.2} & $10.20$ & $21.20$ & $25.80$ & $\mathbf{1.00}$ & $\mathbf{2.41}$ & $\mathbf{3.3

tabella latex delle $\theta$-acc aggregate

In [62]:
cumulative_theta_accuracy_df = pd.concat({v:get_cumulative_theta_accuracies_df(versions_scores_without_category[v]) for _,v in versions if not("ablato" in v)}, axis=0)

cumulative_theta_accuracy_df.index.names = [r"\textbf{Embeddings}", r"\textbf{Oss. (\%)}"]
cumulative_theta_accuracy_df.columns.names = names=[None, None]
cumulative_theta_accuracy_df = cumulative_theta_accuracy_df.rename(
    index={f"p0{i}":f"\\textit{{C.{i}}}" for i in range(8)},
    columns={c[1]:f"\\textbf{{{c[1]}}}" for c in cumulative_theta_accuracy_df.columns}
    ).rename(
        index={int(p*100):f"${int(p*100)}$" for p in percentages},
        columns={
            r"$\theta$-accuracy": r"\textbf{$\theta$-accuracy}",
            "Spread": r"\textbf{Spread}"
        }
    ).rename(
        index={v[1]:f"\\textbf{{{v[1].upper()}}}" for v in versions},
        columns={f"{t}":f"${t}$" for t in thetas}
    )
# cumulative_theta_accuracy_df = cumulative_theta_accuracy_df.map(lambda x: f"${x:.2f}$")
cumulative_theta_accuracy_df = format_cumulative_theta_metrics(cumulative_theta_accuracy_df)

latex_theta_acc_df = cumulative_theta_accuracy_df.to_latex(float_format="%.2f", column_format="cc|ccc|ccc", multirow=True, escape=False)
latex_theta_acc_df = latex_theta_acc_df.replace('\\cline{1-8}\n', '')
latex_theta_acc_df = latex_theta_acc_df.replace('\\midrule\n', '')
latex_theta_acc_df = latex_theta_acc_df.replace(r'\multirow[t]', '\\midrule\n\\multirow')
# latex_theta_acc_df = latex_theta_acc_df.replace(r'\multicolumn{6}{r}', r'\multicolumn{6}{c|}')
latex_theta_acc_df = latex_theta_acc_df.replace(r'\multicolumn{3}{r}', r'\multicolumn{3}{c|}')

latex_theta_acc_df = latex_theta_acc_df.replace(r'\multicolumn{3}{c|}{\textbf{Spread}} \\', r'\multicolumn{3}{c}{\textbf{Spread}} \\')
# latex_theta_acc_df = latex_theta_acc_df.replace(r'\multicolumn{6}{c|}{\textbf{VAE}}', r'\multicolumn{6}{c}{\textbf{VAE}}')

pattern = r"^\s*&\s*&\s*(.*?)\s*\\\\\n\s*(.*?)\s*(?:&\s*)+\\\\"
latex_theta_acc_df = re.sub(pattern, r"\2 & \1 \\\\", latex_theta_acc_df, flags=re.MULTILINE)

print(latex_theta_acc_df)

\begin{tabular}{cc|ccc|ccc}
\toprule
 &  & \multicolumn{3}{c|}{\textbf{$\theta$-accuracy}} & \multicolumn{3}{c}{\textbf{Spread}} \\
\textbf{Embeddings} & \textbf{Oss. (\%)} & \textbf{0.0} & \textbf{0.1} & \textbf{0.2} & \textbf{0.0} & \textbf{0.1} & \textbf{0.2} \\
\midrule
\multirow{5}{*}{\textbf{SAE}} & $10$ & $9.44$ & $21.57$ & $29.49$ & $\mathbf{1.00}$ & $\mathbf{2.72}$ & $\mathbf{3.75}$ \\
 & $30$ & $26.66$ & $44.62$ & $53.75$ & $\mathbf{1.00}$ & $\mathbf{2.21}$ & $3.14$ \\
 & $50$ & $48.86$ & $65.56$ & $73.77$ & $\mathbf{1.00}$ & $1.86$ & $2.65$ \\
 & $70$ & $\mathbf{70.45}$ & $\mathbf{82.07}$ & $87.33$ & $\mathbf{1.00}$ & $1.65$ & $2.34$ \\
 & $100$ & $77.26$ & $85.96$ & $89.50$ & $\mathbf{1.00}$ & $1.55$ & $2.19$ \\
\midrule
\multirow{5}{*}{\textbf{DAE}} & $10$ & $9.84$ & $23.03$ & $\mathbf{30.64}$ & $\mathbf{1.00}$ & $2.75$ & $3.81$ \\
 & $30$ & $\mathbf{28.09}$ & $\mathbf{44.97}$ & $\mathbf{53.80}$ & $\mathbf{1.00}$ & $2.23$ & $\mathbf{3.11}$ \\
 & $50$ & $\mathbf{50.11}$ & $